# 🧠 Unsupervised Learning - Assignment: Session 7

**Topic:** Exploratory Data Analysis (EDA) for Customer Segmentation, Missing Value Audit, Bivariate Distribution Visualizations, Correlation Heatmaps, and K-Means Clustering ($k=3$)

---

### 📋 Assignment Roadmap
1. **Task 1:** Ingest the Kaggle 'Mall Customers' Dataset & Preview the First 10 Observations via Pandas
2. **Task 2:** Data Quality Audit: Check & Enumerate Missing Values in Age, Annual Income, and Spending Score
3. **Task 3:** Bivariate Scatterplot of Annual Income vs. Spending Score (Visualizing Natural Customer Topography)
4. **Task 4:** Seaborn Correlation Heatmap & In-Depth Interpretation of the Dominant Feature Inter-Relationships
5. **Task 5:** Apply K-Means ($k=3$, `random_state=42`), Append the 'Cluster' Feature, and Visualize Segment Boundaries

---

## 1️⃣ Task 1: Ingesting the Customer Segmentation Dataset

### 📖 Conceptual Definition & Dataset Provenance
- **Dataset:** *Mall Customer Segmentation Data* (Kaggle: `vjchoudhary7/customer-segmentation-tutorial-in-python`).
- **Business Objective:** Uncover latent consumer archetypes visiting a shopping center to enable targeted merchandising, loyalty tiering, and tailored promotional campaigns.
- **Features Available:**
  1. `CustomerID`: Unique identifier for each shopper.
  2. `Genre` / `Gender`: Biological gender (`Male`, `Female`).
  3. `Age`: Age of the customer in years (Discrete Numerical).
  4. `Annual Income (k$)`: Annual income in thousands of dollars (Continuous Numerical).
  5. `Spending Score (1-100)`: Behavioral score assigned by the mall management based on spending habits and purchase recency (Ordinal/Continuous Numerical).

In [ ]:
# =====================================================================
# TASK 1: Load Dataset from Kaggle / Local Workspace & Display 10 Rows
# =====================================================================
import os
import pandas as pd

# 1. Load the Mall Customers dataset (local file with fallback to GitHub raw)
csv_path = 'Mall_Customers.csv'
if os.path.exists(csv_path):
    df_mall = pd.read_csv(csv_path)
else:
    url = 'https://raw.githubusercontent.com/MainakRepositor/Datasets/master/Mall_Customers.csv'
    df_mall = pd.read_csv(url)

# Clean column headers to prevent leading/trailing whitespace issues
df_mall.columns = df_mall.columns.str.strip()

print("=" * 75)
print("TASK 1: CUSTOMER SEGMENTATION DATASET PREVIEW (First 10 Rows)")
print("=" * 75)
print(df_mall.head(10).to_string(index=False))

print("\nDataset Dimensions:")
print(f"Total Rows (Customers): {df_mall.shape[0]}")
print(f"Total Columns          : {df_mall.shape[1]} -> {list(df_mall.columns)}")

## 2️⃣ Task 2: Data Quality & Missing Value Audit

### 📖 The Criticality of Missing Value Detection in Clustering
- **Distance Metric Fragility:** Algorithms like K-Means calculate spatial Euclidean distances between points:
  $$d(p, c) = \sqrt{\sum_{d=1}^D (p_d - c_d)^2}$$
  If any coordinate $p_d$ is missing (`NaN` / `null`), the distance formula fails mathematically, causing runtime exceptions.
- In this task, we audit the three core numerical features: `Age`, `Annual Income (k$)`, and `Spending Score (1-100)`.

In [ ]:
# =====================================================================
# TASK 2: Missing Values Audit for Core Segmentation Features
# =====================================================================
# Target columns to inspect
target_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

# Compute missing values
missing_counts = df_mall[target_cols].isnull().sum()
missing_pcts = (df_mall[target_cols].isnull().mean()) * 100

df_missing_summary = pd.DataFrame({
    "Feature Name": target_cols,
    "Total Missing Values": missing_counts.values,
    "Missing Percentage": [f"{pct:.2f}%" for pct in missing_pcts.values],
    "Data Type": [df_mall[col].dtype for col in target_cols]
})

print("=" * 75)
print("TASK 2: MISSING VALUES AUDIT REPORT")
print("=" * 75)
print(df_missing_summary.to_string(index=False))

print("\n" + "=" * 75)
print("SUMMARY PER REQUESTED FEATURE:")
for col in target_cols:
    print(f"  -> {col:<24}: {df_mall[col].isnull().sum()} missing values")
print("=" * 75)
print("Audit Result: ZERO missing values detected across all target attributes.")

## 3️⃣ Task 3: Bivariate Distribution: Annual Income vs. Spending Score

### 📖 Visualizing Unlabeled Topography
- Before executing any clustering algorithm, plotting the raw 2D feature space (`Annual Income` vs. `Spending Score`) allows data scientists to visually inspect the data distribution.
- **Anticipating Natural Geometry:** Inspection of this specific feature space historically reveals a classic **5-cluster pinwheel formation** (low-income low-spenders, low-income high-spenders, middle-income middle-spenders, high-income low-spenders, and high-income high-spenders).

In [ ]:
# =====================================================================
# TASK 3: Scatterplot of Annual Income vs. Spending Score
# =====================================================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 6))

# Create the scatterplot using Seaborn
sns.scatterplot(
    data=df_mall,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    s=90,
    color='#2980b9',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.85,
    ax=ax
)

ax.set_title("Task 3: Customer Distribution: Annual Income vs. Spending Score",
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Annual Income (k$)", fontsize=11, fontweight='bold')
ax.set_ylabel("Spending Score (1 to 100)", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)

# Annotate visual cohorts visible in the raw distribution
ax.text(25, 82, "Low Income,\nHigh Spenders", fontsize=9, fontweight='semibold', color='#c0392b', ha='center')
ax.text(25, 18, "Low Income,\nLow Spenders", fontsize=9, fontweight='semibold', color='#7f8c8d', ha='center')
ax.text(55, 50, "Moderate Income,\nModerate Spenders", fontsize=9, fontweight='semibold', color='#27ae60', ha='center')
ax.text(90, 82, "High Income,\nHigh Spenders (VIP)", fontsize=9, fontweight='semibold', color='#8e44ad', ha='center')
ax.text(90, 18, "High Income,\nLow Spenders", fontsize=9, fontweight='semibold', color='#d35400', ha='center')

plt.tight_layout()
plt.show()

## 4️⃣ Task 4: Numerical Feature Correlation Matrix Heatmap

### 📖 Understanding Pearson Linear Correlation ($r$)
The Pearson correlation coefficient measures linear dependency:

$$r_{xy} = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^n (x_i - \bar{x})^2 \sum_{i=1}^n (y_i - \bar{y})^2}}$$

- $r = +1$: Perfect positive linear correlation.
- $r = 0$: No linear relationship.
- $r = -1$: Perfect negative linear correlation.

We examine the 3 numerical features: `Age`, `Annual Income (k$)`, and `Spending Score (1-100)`.

In [ ]:
# =====================================================================
# TASK 4: Correlation Heatmap using Seaborn
# =====================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Select numerical features and compute Pearson correlation matrix
num_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
corr_matrix = df_mall[num_cols].corr()

# 2. Render Seaborn heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=1.5,
    linecolor='white',
    cbar_kws={'label': 'Pearson Correlation Coefficient (r)'},
    ax=ax
)

ax.set_title("Task 4: Correlation Heatmap of Numerical Customer Features",
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print("=" * 65)
print("CORRELATION VALUES BREAKDOWN:")
print("=" * 65)
print(corr_matrix.round(3))
print("=" * 65)

### 🔍 Detailed Interpretation of Correlation Findings:
1. **Strongest Correlation:** The two features most strongly correlated are **`Age`** and **`Spending Score (1-100)`** with a negative Pearson correlation coefficient of **$r \approx -0.33$**.
   - *Business Meaning:* There is an inverse linear relationship between age and spending propensity. Younger shoppers exhibit higher spending scores, whereas older demographics demonstrate more conservative, restrained purchasing habits.
2. **Near-Zero Correlation:** `Annual Income (k$)` and `Spending Score (1-100)` share an almost negligible correlation of **$r \approx 0.01$**.
   - *Critical Data Science Takeaway:* Earning higher income does **not** automatically translate to higher spending in this mall. Because there is zero linear correlation, linear regression would completely fail—making **unsupervised 2D clustering** the ideal tool to discover non-linear customer segments.

## 5️⃣ Task 5: K-Means Customer Segmentation ($k=3$)

### 📖 Implementation Parameters & Constraints
- **Features Used:** Strictly `['Annual Income (k$)', 'Spending Score (1-100)']`.
- **Algorithm:** `sklearn.cluster.KMeans`.
- **Hyperparameters:** `n_clusters=3`, `random_state=42` (ensures reproducible centroid initialization across all machines).
- **Output Column:** Adds `'Cluster'` to the original DataFrame.

In [ ]:
# =====================================================================
# TASK 5: Apply K-Means (k=3, random_state=42) on Income and Spending
# =====================================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# 1. Select the two clustering features
features_kmeans = ['Annual Income (k$)', 'Spending Score (1-100)']
X_cluster = df_mall[features_kmeans].values

# 2. Instantiate and fit KMeans with k=3 and random_state=42
kmeans_3 = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
df_mall['Cluster'] = kmeans_3.fit_predict(X_cluster)

# Extract centroids
centroids_3 = kmeans_3.cluster_centers_

# 3. Print cluster summary and centroid coordinates
print("=" * 75)
print("TASK 5: K-MEANS (k=3) SEGMENTATION SUMMARY")
print("=" * 75)
cluster_distribution = df_mall['Cluster'].value_counts().sort_index()
for c_id, count in cluster_distribution.items():
    cx, cy = centroids_3[c_id]
    print(f"Cluster {c_id}: {count} customers -> Centroid: [Income: ${cx:.1f}k, Spending Score: {cy:.1f}]")
print("=" * 75)

# Display first 10 rows with the new 'Cluster' column
print("\nFirst 10 Rows of DataFrame with 'Cluster' Column:")
print(df_mall[['CustomerID', 'Annual Income (k$)', 'Spending Score (1-100)', 'Cluster']].head(10).to_string(index=False))

# 4. Plot the 3 Clusters and Centroids
fig, ax = plt.subplots(figsize=(10, 6))
palette_k3 = {0: '#3498db', 1: '#e74c3c', 2: '#2ecc71'}  # Blue, Red, Green
cluster_labels_map = {
    0: "Cluster 0: Low/Moderate Spenders",
    1: "Cluster 1: High Income / High Spenders (VIP)",
    2: "Cluster 2: High Income / Low Spenders (Careful)"
}

for c_id in range(3):
    subset = df_mall[df_mall['Cluster'] == c_id]
    ax.scatter(
        subset['Annual Income (k$)'], subset['Spending Score (1-100)'],
        color=palette_k3[c_id],
        label=f"Cluster {c_id} ({len(subset)} customers)",
        s=80, alpha=0.85, edgecolors='black', linewidth=0.6, zorder=3
    )

# Plot Centroids
ax.scatter(
    centroids_3[:, 0], centroids_3[:, 1],
    marker='*',
    s=380,
    color='#f1c40f',
    edgecolors='black',
    linewidth=1.5,
    label='Centroids (\u03bc)',
    zorder=5
)

# Annotate centroid coordinates
for c_id, (cx, cy) in enumerate(centroids_3):
    ax.annotate(
        f"Centroid {c_id}\n(${cx:.0f}k, {cy:.0f})",
        (cx, cy),
        textcoords="offset points",
        xytext=(10, 8),
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#cccccc", alpha=0.85)
    )

ax.set_title("K-Means Customer Segmentation (k=3) on Annual Income vs. Spending Score",
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Annual Income (k$)", fontsize=11, fontweight='bold')
ax.set_ylabel("Spending Score (1-100)", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", loc="upper right")

plt.tight_layout()
plt.show()

## 🏁 Session 7 Summary & Takeaways

1. **Data Hygiene:** The Kaggle Mall Customers dataset is complete with 0 missing values across all customer attributes.
2. **Linear vs. Non-Linear Independence:** While `Age` and `Spending Score` share a moderate negative linear correlation ($r \approx -0.33$), `Annual Income` and `Spending Score` share virtually zero linear relationship ($r \approx 0.01$). This confirms that customer purchasing power cannot be modeled by simple linear relationships, mandating cluster analysis.
3. **K-Means Partitioning ($k=3$):** Grouping into 3 clusters separates the high-income customers into distinct high-spending vs. low-spending cohorts, with the remaining bulk forming a baseline moderate cohort.
4. **Actionable Business Application:** The resulting clusters give retail managers clear targeting rules (e.g., luxury promotions for Cluster 1 VIPs vs. value deals for Cluster 2).